# Erdos-Straus Modular Sieve -- Google Colab

**Guinea Pig Trench LLC**

---

Modular sieve verification of the [Erdos-Straus conjecture](https://en.wikipedia.org/wiki/Erd%C5%91s%E2%80%93Straus_conjecture).

Based on Salez (2014, arXiv:1406.6307) and Mihnea & Dumitru (2025, arXiv:2509.00128).
Uses ~2.1M residue classes mod G_8 and ~149K prime filters to verify the conjecture
in batches of ~26 billion integers each. Replaces brute-force O(n^2) with O(1) per filtered target.

GitHub: [Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)

In [ ]:
# === Setup & Data Download ===
import os, subprocess, multiprocessing as mp

cpu_count = mp.cpu_count()
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null').read().strip()
print(f'CPU cores: {cpu_count} | RAM: {ram_gb:.1f}GB | GPU: {gpu or "none"}')
print('Tip: Use Runtime > Change runtime type > T4 GPU for more CPU/RAM')

SIEVE_DIR = 'sieve_data'
os.makedirs(SIEVE_DIR, exist_ok=True)

for fname in ['Residues.txt', 'Filters.txt']:
    dest = os.path.join(SIEVE_DIR, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f'{fname} already downloaded ({os.path.getsize(dest)/1e6:.1f}MB)')
        continue
    url = f'https://github.com/esc-paper/erdos-straus/raw/main/section1/resources/{fname}'
    print(f'Downloading {fname}...')
    subprocess.run(['wget', '-q', '-O', dest, url], check=True)
    print(f'  Done: {os.path.getsize(dest)/1e6:.1f}MB')

print('\nSieve data ready.')

In [ ]:
# === Sieve Solver (self-contained) ===
import csv, math, time
import multiprocessing as mp

G_8 = 25_878_772_920

def load_residues(path='sieve_data/Residues.txt'):
    with open(path) as f:
        return list(map(int, f.read().split()))

def load_filters(path='sieve_data/Filters.txt'):
    filters = []
    current_prime = None
    current_residues = []
    with open(path) as f:
        for line in f:
            for token in line.split():
                n = int(token)
                if n == -1:
                    if current_prime is not None:
                        filters.append((current_prime, frozenset(current_residues)))
                    current_prime = None
                    current_residues = []
                elif current_prime is None:
                    current_prime = n
                else:
                    current_residues.append(n)
    if current_prime is not None:
        filters.append((current_prime, frozenset(current_residues)))
    return filters

_w_residues = None
_w_filters = None

def _init_worker(res_path, filt_path):
    global _w_residues, _w_filters
    _w_residues = load_residues(res_path)
    _w_filters = load_filters(filt_path)

def _check_batch(k):
    survivors = []
    for r in _w_residues:
        n = r + k * G_8
        for p, excluded in _w_filters:
            if n % p in excluded:
                break
        else:
            survivors.append(n)
    return survivors

def check_batch(k):
    return (k, _check_batch(k))

def is_prime(n):
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0: return False
        i += 6
    return True

# Verify data loads
res = load_residues()
filt = load_filters()
print(f'Solver loaded: {len(res):,} residues, {len(filt):,} filters')
del res, filt  # free memory before spawning workers

---
## Configure Your Batch Range

| Target | K_START | K_END | Batches |
|--------|---------|-------|----------|
| 10^14  | 0       | 3,864 | 3,865    |
| 10^17  | 0       | 3,864,170 | 3,864,171 |
| 10^18  | 0       | 38,641,709 | 38,641,710 |

**Fleet split for 10^14:**
- Kaggle: 0-966
- **Colab: 967-1933** (this notebook)
- SageMaker: 1934-2900
- Lightning: 2901-3864

In [ ]:
# === Configure ===
K_START = 967      # <-- edit
K_END   = 1933     # <-- edit
WORKERS = max(2, cpu_count)

total_batches = K_END - K_START + 1
n_max = (K_END + 1) * G_8
print(f'Batches: {K_START:,} to {K_END:,} ({total_batches:,} total)')
print(f'Verification up to: ~{n_max:.2e}')
print(f'Workers: {WORKERS}')

In [ ]:
# === Run the Sieve ===
RES_PATH = 'sieve_data/Residues.txt'
FILT_PATH = 'sieve_data/Filters.txt'
checkpoint = f'sieve_results_{K_START}_{K_END}.csv'

# Auto-resume
done_batches = set()
if os.path.exists(checkpoint):
    with open(checkpoint) as f:
        for row in csv.DictReader(f):
            done_batches.add(int(row['batch_k']))
    print(f'Resuming: {len(done_batches):,} batches already done')

remaining = [k for k in range(K_START, K_END + 1) if k not in done_batches]
if not remaining:
    print('All batches done!')
else:
    residue_count = len(load_residues())
    fields = ['batch_k', 'candidates', 'survivors', 'survivor_n', 'is_prime']
    mode = 'a' if done_batches else 'w'
    t0 = time.time()
    processed = 0
    prime_survivors = 0

    print(f'Running {len(remaining):,} batches with {WORKERS} workers...')
    print('=' * 65)

    with open(checkpoint, mode, newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if not done_batches:
            writer.writeheader()

        with mp.Pool(WORKERS, initializer=_init_worker, initargs=(RES_PATH, FILT_PATH)) as pool:
            for k, survivors in pool.imap_unordered(check_batch, remaining, chunksize=1):
                processed += 1
                if survivors:
                    for n in survivors:
                        prime = is_prime(n)
                        writer.writerow({'batch_k': k, 'candidates': residue_count,
                                         'survivors': len(survivors), 'survivor_n': n, 'is_prime': prime})
                        if prime:
                            prime_survivors += 1
                            print(f'\n  *** PRIME SURVIVOR: n={n} (batch {k}) ***')
                else:
                    writer.writerow({'batch_k': k, 'candidates': residue_count,
                                     'survivors': 0, 'survivor_n': '', 'is_prime': ''})

                if processed % 10 == 0 or processed == len(remaining):
                    f.flush()
                    elapsed = time.time() - t0
                    rate = processed / elapsed if elapsed > 0 else 0
                    eta = (len(remaining) - processed) / rate if rate > 0 else 0
                    pct = 100 * processed / len(remaining)
                    print(f'\r  [{pct:5.1f}%] {processed:,}/{len(remaining):,} | '
                          f'{rate:.1f} batch/s | ETA: {eta/60:.0f}m | prime survivors: {prime_survivors}',
                          end='', flush=True)

    total_time = time.time() - t0
    print(f'\n\n{"=" * 65}')
    print(f'Done in {total_time/60:.1f}m ({total_time/3600:.1f}h)')
    print(f'Batches: {processed:,} | Prime survivors: {prime_survivors}')
    if prime_survivors == 0:
        print(f'Conjecture HOLDS for this range')
    print(f'Results: {checkpoint}')

In [ ]:
# === Save to Google Drive ===
import shutil
from google.colab import drive, files

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/erdos_straus'
os.makedirs(drive_dir, exist_ok=True)
shutil.copy2(checkpoint, os.path.join(drive_dir, checkpoint))
print(f'Saved to Drive: {drive_dir}/{checkpoint}')
files.download(checkpoint)

In [ ]:
# === Stats ===
if os.path.exists(checkpoint):
    batches = set()
    survivors = 0
    with open(checkpoint) as f:
        for row in csv.DictReader(f):
            batches.add(int(row['batch_k']))
            if row.get('is_prime', '').lower() == 'true':
                survivors += 1
    k_max = max(batches) if batches else 0
    print(f'Batches completed: {len(batches):,}')
    print(f'Verified up to: ~{(k_max+1)*G_8:.2e}')
    print(f'Prime survivors: {survivors}')
    if survivors == 0:
        print('Conjecture holds for this range!')